In [1]:
# ============================================================
# Notebook    : 09_governance_shap.ipynb
# Description : SHAP analysis across all three Cases, applied
#               only to each Case's best-performing model
#               (per design decision — not baseline vs best,
#               to keep governance scoped to the model that
#               would actually be deployed):
#                 - Case A: ③c LightGBM-Aggregate (AUC 0.7919)
#                 - Case B: B2 LightGBM-Full        (AUC 0.8894)
#                 - Case C: C2 LightGBM-BonusMalus  (AUC 0.7077)
#
#               Models are loaded from saved .txt files (no
#               retraining) — feature columns and split logic
#               must exactly match the notebooks that produced
#               them (03c, 06b, 08b respectively).
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install shap lightgbm pandas numpy matplotlib


# ============================================================
# 1. Common imports
# ============================================================
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt

np.random.seed(42)
SEED = 42

print(f"[CHECK 1] shap version: {shap.__version__}")
print(f"[CHECK 1] lightgbm version: {lgb.__version__}")


# ============================================================
# 2. CASE A — ③c LightGBM-Aggregate
#    Reconstructs the exact test set from 03c, then loads the
#    saved model to compute SHAP values (no retraining).
# ============================================================
print("\n" + "="*60)
print("CASE A — SHAP on ③c (LightGBM-Aggregate)")
print("="*60)

df_a = pd.read_csv("data/fremotor_multi_history_aggregate.csv")

with open("data/sequences/split_idpols.json", "r", encoding="utf-8") as f:
    split_ids = json.load(f)
test_idpols_a = set(split_ids["test"])

df_a_test = df_a[df_a["IDpol"].isin(test_idpols_a)].copy()

NUMERIC_COLS_A = [
    "Expo", "YearGap", "CumClaimCount", "ClaimRateSoFar", "YearsSinceFirstClaimLog",
]
BINARY_COLS_A  = ["PrevLabel", "HasPriorClaim"]
CAT_COLS_A     = ["Usage", "VehType", "VehPower"]
FEATURE_COLS_A = NUMERIC_COLS_A + BINARY_COLS_A + CAT_COLS_A

for col in CAT_COLS_A:
    df_a_test[col] = df_a_test[col].astype("category")

X_test_a = df_a_test[FEATURE_COLS_A].copy()
y_test_a = df_a_test["Label"].copy()

print(f"[CHECK A1] Test set: {X_test_a.shape}")

model_a = lgb.Booster(model_file="data/sequences/lightgbm_aggregate_model.txt")

# LightGBM's saved booster doesn't retain pandas category dtype
# metadata, so category codes must match what was used at train
# time. Since the vocab (sorted unique categories) is
# deterministic from the same source CSV, this reproduces the
# same encoding as 03c without needing to re-run training.
explainer_a = shap.TreeExplainer(model_a)
shap_values_a = explainer_a.shap_values(X_test_a)

# LightGBM binary classification can return a list [neg, pos] or
# a single array depending on version — normalize to the
# "positive class" SHAP values
if isinstance(shap_values_a, list):
    shap_values_a = shap_values_a[1]

print(f"[CHECK A2] SHAP values shape: {shap_values_a.shape}")

# Summary plot — bar (mean |SHAP|), saved to file
plt.figure()
shap.summary_plot(shap_values_a, X_test_a, plot_type="bar", show=False)
plt.title("Case A (③c LightGBM-Aggregate) — SHAP feature importance")
plt.tight_layout()
plt.savefig("data/sequences/shap_case_a_bar.png")
plt.close()
print("Saved: data/sequences/shap_case_a_bar.png")

# Beeswarm plot — direction of effect, saved to file
plt.figure()
shap.summary_plot(shap_values_a, X_test_a, show=False)
plt.title("Case A (③c LightGBM-Aggregate) — SHAP summary")
plt.tight_layout()
plt.savefig("data/sequences/shap_case_a_beeswarm.png")
plt.close()
print("Saved: data/sequences/shap_case_a_beeswarm.png")

# Numeric summary — mean |SHAP| per feature, for the write-up
mean_abs_shap_a = pd.DataFrame({
    "feature": FEATURE_COLS_A,
    "mean_abs_shap": np.abs(shap_values_a).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

print("\n[CHECK A3] Mean |SHAP value| per feature:")
print(mean_abs_shap_a.to_string(index=False))

np.savez(
    "data/sequences/shap_case_a_values.npz",
    shap_values=shap_values_a,
    feature_names=np.array(FEATURE_COLS_A, dtype=object),
)
print("Saved: data/sequences/shap_case_a_values.npz")


# ============================================================
# 3. CASE B — B2 LightGBM-Full
# ============================================================
print("\n" + "="*60)
print("CASE B — SHAP on B2 (LightGBM-Full)")
print("="*60)

df_b = pd.read_csv("data/car_insurance_preprocessed.csv")

NUMERIC_COLS_B = [
    "CREDIT_SCORE", "ANNUAL_MILEAGE", "CHILDREN", "MARRIED", "VEHICLE_OWNERSHIP",
]
CAT_COLS_B = [
    "AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION",
    "INCOME", "VEHICLE_YEAR", "VEHICLE_TYPE",
]
BINARY_FLAG_COLS_B = ["CREDIT_SCORE_was_missing", "ANNUAL_MILEAGE_was_missing"]
BEHAVIORAL_COLS_B  = ["SPEEDING_VIOLATIONS", "DUIS", "PAST_ACCIDENTS"]
FEATURE_COLS_B = NUMERIC_COLS_B + CAT_COLS_B + BINARY_FLAG_COLS_B + BEHAVIORAL_COLS_B

from sklearn.model_selection import train_test_split

X_b = df_b[FEATURE_COLS_B].copy()
y_b = df_b["OUTCOME"].copy()

X_train_b, X_temp_b, y_train_b, y_temp_b = train_test_split(
    X_b, y_b, test_size=0.30, random_state=SEED, stratify=y_b
)
X_val_b, X_test_b, y_val_b, y_test_b = train_test_split(
    X_temp_b, y_temp_b, test_size=0.50, random_state=SEED, stratify=y_temp_b
)

for col in CAT_COLS_B:
    X_test_b[col] = X_test_b[col].astype("category")

print(f"[CHECK B1] Test set: {X_test_b.shape}")

model_b = lgb.Booster(model_file="data/sequences/case_b_lightgbm_full_model.txt")

explainer_b = shap.TreeExplainer(model_b)
shap_values_b = explainer_b.shap_values(X_test_b)
if isinstance(shap_values_b, list):
    shap_values_b = shap_values_b[1]

print(f"[CHECK B2] SHAP values shape: {shap_values_b.shape}")

plt.figure()
shap.summary_plot(shap_values_b, X_test_b, plot_type="bar", show=False)
plt.title("Case B (B2 LightGBM-Full) — SHAP feature importance")
plt.tight_layout()
plt.savefig("data/sequences/shap_case_b_bar.png")
plt.close()
print("Saved: data/sequences/shap_case_b_bar.png")

plt.figure()
shap.summary_plot(shap_values_b, X_test_b, show=False)
plt.title("Case B (B2 LightGBM-Full) — SHAP summary")
plt.tight_layout()
plt.savefig("data/sequences/shap_case_b_beeswarm.png")
plt.close()
print("Saved: data/sequences/shap_case_b_beeswarm.png")

mean_abs_shap_b = pd.DataFrame({
    "feature": FEATURE_COLS_B,
    "mean_abs_shap": np.abs(shap_values_b).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

print("\n[CHECK B3] Mean |SHAP value| per feature:")
print(mean_abs_shap_b.to_string(index=False))

np.savez(
    "data/sequences/shap_case_b_values.npz",
    shap_values=shap_values_b,
    feature_names=np.array(FEATURE_COLS_B, dtype=object),
)
print("Saved: data/sequences/shap_case_b_values.npz")

# flag for use in notebook 11 (fairness audit) — save X_test_b
# with GENDER/RACE alongside so the cohort labels line up
# exactly with these SHAP rows
X_test_b.to_csv("data/sequences/case_b_X_test_for_fairness.csv", index=False)
y_test_b.to_csv("data/sequences/case_b_y_test_for_fairness.csv", index=False)
print("Saved: X_test/y_test snapshot for notebook 11 (fairness audit)")


# ============================================================
# 4. CASE C — C2 LightGBM-BonusMalus
# ============================================================
print("\n" + "="*60)
print("CASE C — SHAP on C2 (LightGBM-BonusMalus)")
print("="*60)

df_c = pd.read_csv("data/fremtpl2_preprocessed.csv")

NUMERIC_COLS_C = ["Exposure", "VehPower", "VehAge", "DrivAge", "Density", "BonusMalus"]
CAT_COLS_C     = ["VehBrand", "VehGas", "Area", "Region"]
FEATURE_COLS_C = NUMERIC_COLS_C + CAT_COLS_C

X_c = df_c[FEATURE_COLS_C].copy()
y_c = df_c["Label"].copy()

X_train_c, X_temp_c, y_train_c, y_temp_c = train_test_split(
    X_c, y_c, test_size=0.30, random_state=SEED, stratify=y_c
)
X_val_c, X_test_c, y_val_c, y_test_c = train_test_split(
    X_temp_c, y_temp_c, test_size=0.50, random_state=SEED, stratify=y_temp_c
)

for col in CAT_COLS_C:
    X_test_c[col] = X_test_c[col].astype("category")

print(f"[CHECK C1] Test set: {X_test_c.shape}")

# SHAP on the full 101,702-row test set can be slow for
# TreeExplainer with many categorical splits — sample down for
# speed if needed, but try full set first since LightGBM
# TreeExplainer is usually fast enough
model_c = lgb.Booster(model_file="data/sequences/case_c_lightgbm_bonusmalus_model.txt")

explainer_c = shap.TreeExplainer(model_c)
shap_values_c = explainer_c.shap_values(X_test_c)
if isinstance(shap_values_c, list):
    shap_values_c = shap_values_c[1]

print(f"[CHECK C2] SHAP values shape: {shap_values_c.shape}")

plt.figure()
shap.summary_plot(shap_values_c, X_test_c, plot_type="bar", show=False)
plt.title("Case C (C2 LightGBM-BonusMalus) — SHAP feature importance")
plt.tight_layout()
plt.savefig("data/sequences/shap_case_c_bar.png")
plt.close()
print("Saved: data/sequences/shap_case_c_bar.png")

plt.figure()
shap.summary_plot(shap_values_c, X_test_c, show=False)
plt.title("Case C (C2 LightGBM-BonusMalus) — SHAP summary")
plt.tight_layout()
plt.savefig("data/sequences/shap_case_c_beeswarm.png")
plt.close()
print("Saved: data/sequences/shap_case_c_beeswarm.png")

mean_abs_shap_c = pd.DataFrame({
    "feature": FEATURE_COLS_C,
    "mean_abs_shap": np.abs(shap_values_c).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

print("\n[CHECK C3] Mean |SHAP value| per feature:")
print(mean_abs_shap_c.to_string(index=False))

np.savez(
    "data/sequences/shap_case_c_values.npz",
    shap_values=shap_values_c,
    feature_names=np.array(FEATURE_COLS_C, dtype=object),
)
print("Saved: data/sequences/shap_case_c_values.npz")


# ============================================================
# 5. Cross-Case comparison — does SHAP rank order match the
#    LightGBM gain-based feature importance seen in 03c/06b/08b?
# ============================================================
print("\n" + "="*60)
print("CROSS-CASE SHAP SUMMARY")
print("="*60)

print("\nCase A top 3 (SHAP):")
print(mean_abs_shap_a.head(3).to_string(index=False))
print("\nCase B top 3 (SHAP):")
print(mean_abs_shap_b.head(3).to_string(index=False))
print("\nCase C top 3 (SHAP):")
print(mean_abs_shap_c.head(3).to_string(index=False))


# ============================================================
# 6. Summary
# ============================================================
print("\n===== SHAP Governance Summary =====")
print(f"Case A (③c) top feature (SHAP): {mean_abs_shap_a.iloc[0]['feature']}")
print(f"Case B (B2)  top feature (SHAP): {mean_abs_shap_b.iloc[0]['feature']}")
print(f"Case C (C2)  top feature (SHAP): {mean_abs_shap_c.iloc[0]['feature']}")
print(f"\nOutputs saved to data/sequences/:")
print(f"  shap_case_{{a,b,c}}_bar.png, shap_case_{{a,b,c}}_beeswarm.png")
print(f"  shap_case_{{a,b,c}}_values.npz")
print(f"  case_b_X_test_for_fairness.csv, case_b_y_test_for_fairness.csv (for notebook 11)")
print("=====================================")

C:\Users\User\anaconda3\envs\diceml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[CHECK 1] shap version: 0.49.1
[CHECK 1] lightgbm version: 4.6.0

CASE A — SHAP on ③c (LightGBM-Aggregate)
[CHECK A1] Test set: (54755, 10)


C:\Users\User\anaconda3\envs\diceml\lib\site-packages\shap\explainers\_tree.py:586: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
C:\Users\User\AppData\Local\Temp\ipykernel_4672\2596320608.py:94: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values_a, X_test_a, plot_type="bar", show=False)


[CHECK A2] SHAP values shape: (54755, 10)
Saved: data/sequences/shap_case_a_bar.png


C:\Users\User\AppData\Local\Temp\ipykernel_4672\2596320608.py:103: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values_a, X_test_a, show=False)


...

Saved: data/sequences/shap_case_b_beeswarm.png

[CHECK B3] Mean |SHAP value| per feature:
                   feature  mean_abs_shap
        DRIVING_EXPERIENCE       1.066845
         VEHICLE_OWNERSHIP       0.504611
              VEHICLE_YEAR       0.405529
                    GENDER       0.230058
            PAST_ACCIDENTS       0.118756
                   MARRIED       0.113406
            ANNUAL_MILEAGE       0.089927
              CREDIT_SCORE       0.063045
       SPEEDING_VIOLATIONS       0.047048
                       AGE       0.029317
                  CHILDREN       0.026853
                      RACE       0.010198
                      DUIS       0.008916
                 EDUCATION       0.008425
  CREDIT_SCORE_was_missing       0.005783
ANNUAL_MILEAGE_was_missing       0.004862
                    INCOME       0.004729
              VEHICLE_TYPE       0.003245
Saved: data/sequences/shap_case_b_values.npz
Saved: X_test/y_test snapshot for notebook 11 (fairness audit)

CAS

C:\Users\User\anaconda3\envs\diceml\lib\site-packages\shap\explainers\_tree.py:586: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
C:\Users\User\AppData\Local\Temp\ipykernel_4672\2596320608.py:254: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values_c, X_test_c, plot_type="bar", show=False)


[CHECK C2] SHAP values shape: (101702, 10)
Saved: data/sequences/shap_case_c_bar.png


C:\Users\User\AppData\Local\Temp\ipykernel_4672\2596320608.py:262: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values_c, X_test_c, show=False)


Saved: data/sequences/shap_case_c_beeswarm.png

[CHECK C3] Mean |SHAP value| per feature:
   feature  mean_abs_shap
  Exposure       0.442304
BonusMalus       0.198513
    VehAge       0.162377
   DrivAge       0.133069
    Region       0.120249
  VehBrand       0.081293
  VehPower       0.067060
   Density       0.061679
    VehGas       0.038774
      Area       0.009392
Saved: data/sequences/shap_case_c_values.npz

CROSS-CASE SHAP SUMMARY

Case A top 3 (SHAP):
       feature  mean_abs_shap
      VehPower       0.775366
       VehType       0.324831
ClaimRateSoFar       0.272629

Case B top 3 (SHAP):
           feature  mean_abs_shap
DRIVING_EXPERIENCE       1.066845
 VEHICLE_OWNERSHIP       0.504611
      VEHICLE_YEAR       0.405529

Case C top 3 (SHAP):
   feature  mean_abs_shap
  Exposure       0.442304
BonusMalus       0.198513
    VehAge       0.162377

===== SHAP Governance Summary =====
Case A (③c) top feature (SHAP): VehPower
Case B (B2)  top feature (SHAP): DRIVING_EXPERIENC